# Text-To-Image Generator in Python 

* Uses the automatic111 app  which must be also running
* Automatic 111 exposes an API that can be called from python
* This example uses a Gradio interface
* based this on [Make a Stable Diffusion easy interface with python](https://civitai.com/articles/4090/make-a-stable-diffusion-easy-interface-with-python)

In [7]:
import os
from dotenv import load_dotenv
from huggingface_hub import login





In [8]:
# following https://civitai.com/articles/4090/make-a-stable-diffusion-easy-interface-with-python
# Import the libraries
import gradio as gr
import requests
import json
from PIL import Image
from io import BytesIO
import base64
import datetime

In [9]:
load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')


In [10]:
textToImg_url = "http://127.0.0.1:7860/sdapi/v1/txt2img"
outputDir = "G:/GenerativeAIOutput/pythonSD"
model_juggernaut = "juggernautXL_v8Rundiffusion.safetensors [aeb7e9e689]"
default_negative_prompt = "blurry, low quality, bad anatomy"



In [11]:
# Define the function to call the API
# Must start Automatic 1111 web server before running this code
def call_api(prompt, negative_prompt):
    # Define the URL of the API endpoint
    url = textToImg_url
    data = {
        "prompt": prompt,
        "negative_prompt": default_negative_prompt + negative_prompt,
        "steps": 30, #default is 20
        "sampler_name": "DPM++ 2M Karras", #default is Euler 
        "cfg_scale": 7,
        "seed": -1, # -1 for random seed
        "width": 1024, #default is 512
        "height": 1024,
        "override_settings": {
        "sd_model_checkpoint": model_juggernaut
    }
}






    # Convert the data to JSON format
    json_data = json.dumps(data)

    # Set the headers for the request
    headers = {'Content-Type': 'application/json'}

    # Send the POST request to the API
    response = requests.post(url, data=json_data, headers=headers)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
       # Decode the JSON response
        json_response = response.json()

        # Extract the base64 image data from the response
        image_data = json_response.get('images', [''])[0]

        # Decode the base64 image data
        image_bytes = base64.b64decode(image_data)

        # Open the image using PIL
        image = Image.open(BytesIO(image_bytes))
        current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")    
        image.save(f"{outputDir}/{current_time}.jpg")  # Save the image to a file
        # Display the image
        return image
    else:
        # Return an error message if the request was not successful
        return "Error:", response.status_code

In [12]:
# Create the Gradio interface
iface = gr.Interface(fn=call_api, 
                     inputs=[gr.Textbox(label="Prompt"),
                             gr.Textbox(label="Negative Prompt")],
                     outputs="image")

# Launch the interface
iface.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
